<a href="https://colab.research.google.com/github/chittaaparna-hub/AI-agents-/blob/feature-jobsearch/AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Newly added code
!pip install -q \
    -U openai \
    -U langchain \
    -U langchain-core \
    -U langchain-community \
    -U langchain-openai \
    -U langchain-chroma \
    -U chromadb \
    -U langgraph \
    -U mem0ai \
    -U duckduckgo-search

In [ ]:
# Newly added dependencies
!pip install -q uv

%pip install requests==2.34.2 opentelemetry-api==1.45.1 opentelemetry-sdk==1.45.1

In [ ]:
!pip install -q -U \
    openai \
    langchain \
    langchain-core \
    langchain-openai \
    langchain-community \
    langgraph \
    langchain-chroma \
    chromadb \
    mem0ai \
    duckduckgo-search
%pip install requests==2.34.2 opentelemetry-api==1.45.1 opentelemetry-sdk==1.45.1

In [ ]:
import json
!pip install langchainhub
#import langchainhub as hub

from langchain.agents import create_agent

from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

from langchain_openai import AzureChatOpenAI, ChatOpenAI

from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict
from IPython.display import Image, display


from typing import List, Dict
import requests
import json
from duckduckgo_search import DDGS
import os

from openai import OpenAI


In [ ]:
# STEP 1.2 IMPORT THE NECESSARY MODULES

# For loading the OpenAI API key safely into the colab environment
import os
from google.colab import userdata

# For Extracting the text from the URL
import requests
from bs4 import BeautifulSoup
!pip install beautifulsoup4

# For creating the agent
from langchain.agents import create_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI

In [ ]:
# Initialize OpenAI client
#client = OpenAI(api_key=API_KEY, base_url=OPENAI_API_BASE)

In [ ]:
# STEP 1.3 LOAD THE OPEN AI API KEY
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [ ]:
# STEP 2: DEFINE A FUNCTION TO EXTRACT TEXT FROM URL
def extract_website_text(url):
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")
  return soup.get_text()


In [ ]:
#DEFINE THE FUNCTION FOR PROCESSING THE USER INPUT
!pip install ddgs
def duckduckgo_search_urls(query: str, max_results: int = 10) -> List[Dict[str, str]]:
    """Run a DuckDuckGo search and return list of {title, snippet, url}."""
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title": r.get("title"),
                    "snippet": r.get("body"),
                    "url": r.get("url")
                })
    except Exception as e:
        print(f"[duckduckgo_search_urls] search failed: {e}")
    return results

In [ ]:
# CREATE THE TOOL FOR THE AGENT USING THE FUNCTION CREATED

from langchain_core.tools import Tool

tool = Tool(
    name="duckduckgo_search",
    func=duckduckgo_search_urls,
    description="Use to search the web for factual information (DuckDuckGo). Input: natural language query. Output: short summary of top results."
)

In [ ]:
from langchain.agents import create_agent

# INITIALIZE THE AGENT
agent = create_agent(
    model=llm,
    tools=[tool],
    system_prompt=(
     """You are a job search assistant.
Format every answer in clean Markdown:
- Start with a one-line summary.
- List each job as: **Job Title** - Company - Location
  followed by 1-2 short bullets (key skills, salary if available) and the link.
- Use at most 8 results, no long paragraphs.
- End with 2-3 short tips for entry-level applicants."""
    ),
    debug=True
)

In [ ]:
# Ask the agent a question
question = "Data sceintist jobs along with location"

response = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

# Display the final answer
answer = response["messages"][-1].content
print(answer)